# Paired Wilcoxon on per-bug AutoFL ranks

For each **bug**, we compare the **best** `autofl_rank` of two setups on the **same** bugs (same length, same order). We call **`scipy.stats.wilcoxon(rank_target, rank_baseline, ...)`** with those two aligned vectors—no manual difference array in our code. SciPy’s two-argument form runs the signed-rank procedure on the paired comparison (equivalent to forming `target − baseline` internally).  
Lower rank is better, so systematic **improvement** for the target shows up as target ranks **smaller** than baseline on the same bugs.

**Effect size (paired):**  
**P(target better)** = (wins + 0.5 × ties) / n on common bugs, where a *win* means `rank_target < rank_baseline`. Values **above 0.5** mean the target wins more often; **below 0.5** means the baseline wins more often; **0.5** means balance (ties split 50/50).  
The independent-sample **Vargha–Delaney A₁₂** is not the right tool for this design.

This notebook replaces `vargha-delaney.ipynb`, which used **Mann–Whitney** on two columns of ranks for the same bugs (inappropriate for paired samples; the old table also mixed up U and p in the saved output).

`p` from Wilcoxon: use **`compare_with_baselines(..., wilcoxon_alternative=...)`** with **`"two-sided"`** (default), or **`"less"`** for a **one-sided** test that the target has **lower** (better) ranks than the baseline: H₁: median(`target − baseline`) < 0. Use **`"greater"`** to test the opposite (target worse). SciPy is called with the same `wilcoxon(a, b, alternative=...)`, `d = a − b` = target minus baseline. We use **`zero_method="pratt"`** (Pratt 1959).

**Conditional (“disputed”) Wilcoxon:** with **`run_conditional_wilcoxon=True`**, the notebook also runs the same signed-rank test **only** on bugs where `rank_target ≠ rank_baseline` (no paired tie). That **conditions** the test on there being a disagreement. The printed **p** is a **conditional** p-value (it is *not* the same as the unconditional test on all bugs). Use it to inspect **where the two systems disagree**; for global claims, use the unfiltered Wilcoxon row. A row **`n` disputed** reports how many bugs enter that test for each target–baseline pair.


In [13]:
import json

import numpy as np
from scipy.stats import wilcoxon


def _wilcoxon_paired_raw(
    a: np.ndarray,
    b: np.ndarray,
    alternative: str = "two-sided",
) -> tuple[float, float]:
    """Signed-rank test: scipy `wilcoxon(a, b)` with d = a − b. alternative: 'two-sided' | 'less' | 'greater'.

    With a=target, b=baseline and lower rank better: 'less' tests H1: median(d) < 0 (target better).
    """
    if alternative not in ("two-sided", "less", "greater"):
        raise ValueError("alternative must be 'two-sided', 'less', or 'greater'")
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.size == 0 or a.shape != b.shape:
        return float("nan"), float("nan")
    if np.array_equal(a, b):
        return 0.0, 1.0
    try:
        res = wilcoxon(
            a, b, zero_method="pratt", method="asymptotic", alternative=alternative
        )
    except TypeError:
        try:
            res = wilcoxon(
                a, b, zero_method="pratt", mode="asymptotic", alternative=alternative
            )
        except TypeError:
            if alternative != "two-sided":
                raise TypeError(
                    "Your SciPy is too old: upgrade scipy>=1.7 for Wilcoxon "
                    "alternative=, or use wilcoxon_alternative='two-sided'."
                ) from None
            res = wilcoxon(a, b, zero_method="pratt", method="asymptotic")
    p_value = float(res.pvalue)
    if p_value != p_value:  # SciPy can return NaN p when a and b are fully tied
        p_value = 1.0
    return float(res.statistic), p_value


def paired_stochastic_superiority(r_target: dict[str, float], r_baseline: dict[str, float]) -> float:
    """
    P(target is better) with lower rank = better: (# wins + 0.5 * # ties) / n on common bugs.
    0.5 = no net advantage; above 0.5 = target wins more often than baseline.
    """
    common = sorted(set(r_target) & set(r_baseline))
    if not common:
        return 0.5
    wins = ties = 0.0
    for bug in common:
        t, b = r_target[bug], r_baseline[bug]
        if t < b:
            wins += 1.0
        elif t == b:
            ties += 1.0
    n = float(len(common))
    return (wins + 0.5 * ties) / n


def _fmt_pvalue(p: float) -> str:
    """Avoid printing 0.000 for underflow; keep normal cases to 3 decimals."""
    if p != p:  # NaN
        return "nan"
    if p < 1e-12:
        return "<1e-12"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.3f}"


def compare_paired_ranks_wilcoxon(
    r_target: dict[str, float],
    r_baseline: dict[str, float],
    wilcoxon_alternative: str = "two-sided",
) -> tuple[float, float, float]:
    """
    Returns (wilcoxon_statistic, p_value, P_target_better). For p-value meaning see
    wilcoxon_alternative: 'two-sided' or one-sided 'less' / 'greater' (target better / worse).
    """
    common = sorted(set(r_target) & set(r_baseline))
    a_res = np.array([r_target[cb] for cb in common], dtype=float)
    b_res = np.array([r_baseline[cb] for cb in common], dtype=float)
    p_better = paired_stochastic_superiority(r_target, r_baseline)
    w_stat, p_value = _wilcoxon_paired_raw(
        a_res, b_res, alternative=wilcoxon_alternative
    )
    return w_stat, p_value, p_better


def compare_paired_ranks_wilcoxon_disputed(
    r_target: dict[str, float],
    r_baseline: dict[str, float],
    wilcoxon_alternative: str = "two-sided",
) -> tuple[float, float, float, int]:
    """
    Conditional (\"disputed\") Wilcoxon: use only bugs with rank_target != rank_baseline
    (no ties on the pair — target and baseline differ). Same scipy.test as above on the
    filtered pairs. p-value is *conditional* on having a disagreement; interpret with care.

    Returns (w_stat, p_value, P_target_better_on_disputed, n_disputed).
    On the disputed set, P_better = (# target wins) / n_disputed (no 0.5 tie term needed).
    """
    common = sorted(set(r_target) & set(r_baseline))
    a_res = np.array([r_target[cb] for cb in common], dtype=float)
    b_res = np.array([r_baseline[cb] for cb in common], dtype=float)
    mask = ~np.isclose(a_res, b_res, rtol=0, atol=0, equal_nan=True)
    a_d = a_res[mask]
    b_d = b_res[mask]
    n_d = int(a_d.size)
    if n_d == 0:
        return float("nan"), float("nan"), 0.5, 0
    wins = float(np.sum(a_d < b_d))
    p_better_d = wins / n_d
    w_stat, p_value = _wilcoxon_paired_raw(
        a_d, b_d, alternative=wilcoxon_alternative
    )
    return w_stat, p_value, p_better_d, n_d


def extract_best_ranks(data: dict) -> dict[str, float]:
    bug_ranks: dict[str, float] = {}
    for bug, methods in data.get("buggy_methods", {}).items():
        ranks: list[float] = []
        for m in methods.values():
            if not isinstance(m, dict):
                continue
            r = m.get("autofl_rank")
            if r is not None:
                ranks.append(float(r))
        if ranks:
            bug_ranks[bug] = min(ranks)
    return bug_ranks


def get_ranks(exp_name: str, gpt_version: str, repetition: int) -> dict[str, float]:
    score_path_json = f"../combined_fl_results/{exp_name}/{gpt_version}_R{repetition}_full_light.json"
    with open(score_path_json, "r", encoding="utf-8") as f:
        data: dict = json.load(f)
    return extract_best_ranks(data)


def count_acc_at_1(bug_best_ranks: dict[str, float]) -> int:
    """Acc@1: number of bugs whose best autofl_rank is 1 (top-1)."""
    return sum(1 for r in bug_best_ranks.values() if float(r) <= 1.0)


def _pvalue_caption_wilcoxon(alt: str) -> str:
    if alt == "two-sided":
        return "p-value (two-sided Wilcoxon)"
    if alt == "less":
        return "p-value (one-sided Wilcoxon, target better: median(target−baseline)<0)"
    if alt == "greater":
        return "p-value (one-sided Wilcoxon, target worse: median(target−baseline)>0)"
    return f"p-value (Wilcoxon, alternative={alt!r})"


def _caption_conditional_wilcoxon(alt: str) -> str:
    """Short LaTeX comment for conditional (disputed-only) p row."""
    if alt == "two-sided":
        return "cond. p, disputed only (rank_t≠rank_b, two-sided Wilcoxon)"
    if alt == "less":
        return "cond. p, disputed only (one-sided 'less' = target better)"
    if alt == "greater":
        return "cond. p, disputed only (one-sided 'greater' = target worse)"
    return f"cond. p, disputed only (alt={alt!r})"


def compare_with_baselines(
    target_exp: str,
    gpt_version: str = "gpt-3.5-turbo-0125",
    wilcoxon_alternative: str = "two-sided",
    run_conditional_wilcoxon: bool = True,
) -> None:
    """Per repetition, print: Acc@1, P(target better), p-values, optional conditional p, …

    Acc@1 row column order: target_exp | current | autofl_onlyft | autofl.
    Paired test rows: current | autofl_onlyft | autofl (target vs each baseline).

    wilcoxon_alternative: "two-sided" (default) | "less" (test target has lower / better rank) | "greater".
    run_conditional_wilcoxon: if True, add rows for Wilcoxon on bugs where rank_target != rank_baseline only.
    """
    for rep in (1, 5):
        rfl_cr = get_ranks("current", gpt_version, rep)
        afl_onlyft = get_ranks("autofl_onlyft", gpt_version, rep)
        afl = get_ranks("autofl", gpt_version, rep)
        target_ranks = get_ranks(target_exp, gpt_version, rep)
        wa = wilcoxon_alternative
        w_cr, pval_cr, sup_cr = compare_paired_ranks_wilcoxon(
            target_ranks, rfl_cr, wilcoxon_alternative=wa
        )
        w_oft, pval_oft, sup_oft = compare_paired_ranks_wilcoxon(
            target_ranks, afl_onlyft, wilcoxon_alternative=wa
        )
        w_afl, pval_afl, sup_afl = compare_paired_ranks_wilcoxon(
            target_ranks, afl, wilcoxon_alternative=wa
        )
        if run_conditional_wilcoxon:
            _, pcd_cr, pbd_cr, nd_cr = compare_paired_ranks_wilcoxon_disputed(
                target_ranks, rfl_cr, wilcoxon_alternative=wa
            )
            _, pcd_oft, pbd_oft, nd_oft = compare_paired_ranks_wilcoxon_disputed(
                target_ranks, afl_onlyft, wilcoxon_alternative=wa
            )
            _, pcd_afl, pbd_afl, nd_afl = compare_paired_ranks_wilcoxon_disputed(
                target_ranks, afl, wilcoxon_alternative=wa
            )
        n_t = count_acc_at_1(target_ranks)
        n_cr = count_acc_at_1(rfl_cr)
        n_oft = count_acc_at_1(afl_onlyft)
        n_afl = count_acc_at_1(afl)
        acc_row = (
            f"& {n_t} & {n_cr} & {n_oft} & {n_afl}  % Acc@1 count "
            f"(target | current | onlyft | autofl), R{rep}"
        )
        sup_row = f"& {sup_cr:.3f} & {sup_oft:.3f} & {sup_afl:.3f}  % P(target better), R{rep}"
        pval_row = (
            f"& {_fmt_pvalue(pval_cr)} & {_fmt_pvalue(pval_oft)} & {_fmt_pvalue(pval_afl)}  "
            f"% {_pvalue_caption_wilcoxon(wilcoxon_alternative)}, R{rep}"
        )
        if run_conditional_wilcoxon:
            pbd_row = (
                f"& {pbd_cr:.3f} & {pbd_oft:.3f} & {pbd_afl:.3f}  "
                f"% P(target better | rank_t ≠ rank_b), R{rep}"
            )
            pcond_row = (
                f"& {_fmt_pvalue(pcd_cr)} & {_fmt_pvalue(pcd_oft)} & {_fmt_pvalue(pcd_afl)}  "
                f"% {_caption_conditional_wilcoxon(wa)}, R{rep}"
            )
            nd_row = (
                f"& {nd_cr} & {nd_oft} & {nd_afl}  % n bugs disputed (|rank_t−rank_b|>0), R{rep}"
            )
            extra = (pbd_row, pcond_row, nd_row)
        else:
            extra = ()
        #w_row = f"& {w_cr:.3f} & {w_oft:.3f} & {w_afl:.3f}  % Wilcoxon stat, R{rep}"
        print("\n".join((acc_row, sup_row, pval_row) + extra))


In [10]:
# Use your own LaTeX row macro in the .tex file (e.g. \newcommand{\tw}[1]{#1-day window}).
print(r"% ---- 30-day time window: replace leading comment with e.g. \tw{30} in the paper table ----")
#compare_with_baselines("reportfl")
#print(r"% ---- 60-day ----")
#compare_with_baselines("reportfl_60days")
#print(r"% ---- 90-day ----")
compare_with_baselines("reportfl_90days")

% ---- 30-day time window: replace leading comment with e.g. \tw{30} in the paper table ----
& 127 & 106 & 95 & 124  % Acc@1 count (target | current | onlyft | autofl), R1
& 0.529 & 0.611 & 0.323  % P(target better), R1
& 0.326 & 5.64e-05 & <1e-12  % p-value (two-sided), R1
& 137 & 138 & 114 & 130  % Acc@1 count (target | current | onlyft | autofl), R5
& 0.497 & 0.554 & 0.380  % P(target better), R5
& 0.649 & 0.034 & 8.77e-11  % p-value (two-sided), R5


In [14]:
# Use your own LaTeX row macro in the .tex file (e.g. \newcommand{\tw}[1]{#1-day window}).
print(r"% ---- 30-day time window: replace leading comment with e.g. \tw{30} in the paper table ----")
#compare_with_baselines("reportfl")
#print(r"% ---- 60-day ----")
#compare_with_baselines("reportfl_60days")
#print(r"% ---- 90-day ----")
compare_with_baselines("reportfl_90days", wilcoxon_alternative="less")

% ---- 30-day time window: replace leading comment with e.g. \tw{30} in the paper table ----
& 127 & 106 & 95 & 124  % Acc@1 count (target | current | onlyft | autofl), R1
& 0.529 & 0.611 & 0.323  % P(target better), R1
& 0.163 & 2.82e-05 & 1.000  % p-value (one-sided Wilcoxon, target better: median(target−baseline)<0), R1
& 137 & 138 & 114 & 130  % Acc@1 count (target | current | onlyft | autofl), R5
& 0.497 & 0.554 & 0.380  % P(target better), R5
& 0.675 & 0.017 & 1.000  % p-value (one-sided Wilcoxon, target better: median(target−baseline)<0), R5


In [11]:
print(r"% ---- gpt-4.1-mini, 30 / 60 / 90 days ----")
#compare_with_baselines("reportfl", gpt_version="gpt-4.1-mini-2025-04-14")
print(r"% ---- 60d ----")
compare_with_baselines("reportfl_60days", gpt_version="gpt-4.1-mini-2025-04-14")
#print(r"% ---- 90d ----")
#compare_with_baselines("reportfl_90days", gpt_version="gpt-4.1-mini-2025-04-14")

% ---- gpt-4.1-mini, 30 / 60 / 90 days ----
% ---- 60d ----
& 184 & 179 & 139 & 130  % Acc@1 count (target | current | onlyft | autofl), R1
& 0.500 & 0.580 & 0.536  % P(target better), R1
& 0.975 & 1.38e-04 & 0.686  % p-value (two-sided), R1
& 193 & 182 & 155 & 138  % Acc@1 count (target | current | onlyft | autofl), R5
& 0.516 & 0.571 & 0.568  % P(target better), R5
& 0.494 & 3.23e-04 & 0.059  % p-value (two-sided), R5


In [15]:
print(r"% ---- gpt-4.1-mini, 30 / 60 / 90 days ----")
#compare_with_baselines("reportfl", gpt_version="gpt-4.1-mini-2025-04-14")
print(r"% ---- 60d ----")
compare_with_baselines("reportfl_60days", gpt_version="gpt-4.1-mini-2025-04-14", wilcoxon_alternative="less")
#print(r"% ---- 90d ----")
#compare_with_baselines("reportfl_90days", gpt_version="gpt-4.1-mini-2025-04-14")

% ---- gpt-4.1-mini, 30 / 60 / 90 days ----
% ---- 60d ----
& 184 & 179 & 139 & 130  % Acc@1 count (target | current | onlyft | autofl), R1
& 0.500 & 0.580 & 0.536  % P(target better), R1
& 0.487 & 6.92e-05 & 0.657  % p-value (one-sided Wilcoxon, target better: median(target−baseline)<0), R1
& 193 & 182 & 155 & 138  % Acc@1 count (target | current | onlyft | autofl), R5
& 0.516 & 0.571 & 0.568  % P(target better), R5
& 0.247 & 1.62e-04 & 0.030  % p-value (one-sided Wilcoxon, target better: median(target−baseline)<0), R5
